# DCGAN on CIFAR-10 with ResNet18 Evaluation

Train a convolutional GAN on CIFAR-10 and evaluate generated images using a pretrained ResNet18 classifier.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt


In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
BATCH_SIZE=128
LATENT_DIM=100
EPOCHS=10

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

dataset=torchvision.datasets.CIFAR10(root='./data',train=True,download=True,transform=transform)
loader=torch.utils.data.DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True)

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.ConvTranspose2d(LATENT_DIM,256,4,1,0,bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256,128,4,2,1,bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128,64,4,2,1,bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64,3,4,2,1,bias=False),
            nn.Tanh()
        )
    def forward(self,z):
        return self.net(z)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(3,64,4,2,1,bias=False),
            nn.LeakyReLU(0.2,True),
            nn.Conv2d(64,128,4,2,1,bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2,True),
            nn.Conv2d(128,256,4,2,1,bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2,True),
            nn.Conv2d(256,1,4,1,0,bias=False),
            nn.Sigmoid()
        )
    def forward(self,x):
        return self.net(x).view(-1,1)

In [ ]:
G=Generator().to(device)
D=Discriminator().to(device)
criterion=nn.BCELoss()
optG=optim.Adam(G.parameters(),lr=0.0002,betas=(0.5,0.999))
optD=optim.Adam(D.parameters(),lr=0.0002,betas=(0.5,0.999))

In [ ]:
def show_samples(generator):
    generator.eval()
    with torch.no_grad():
        z=torch.randn(16,LATENT_DIM,1,1,device=device)
        imgs=generator(z).cpu()
    grid=make_grid(imgs,nrow=4,normalize=True)
    plt.figure(figsize=(6,6))
    plt.imshow(grid.permute(1,2,0))
    plt.axis('off')
    plt.show()
    generator.train()

In [ ]:
g_losses=[]
d_losses=[]

for epoch in range(EPOCHS):
    for real,_ in loader:
        real=real.to(device)
        b=real.size(0)

        real_y=torch.ones(b,1,device=device)
        fake_y=torch.zeros(b,1,device=device)

        noise=torch.randn(b,LATENT_DIM,1,1,device=device)
        fake=G(noise)

        d_loss=criterion(D(real),real_y)+criterion(D(fake.detach()),fake_y)
        optD.zero_grad(); d_loss.backward(); optD.step()

        noise=torch.randn(b,LATENT_DIM,1,1,device=device)
        fake=G(noise)
        g_loss=criterion(D(fake),real_y)
        optG.zero_grad(); g_loss.backward(); optG.step()

    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())
    print(epoch+1,g_loss.item(),d_loss.item())
    show_samples(G)

In [ ]:
plt.plot(g_losses,label="Generator")
plt.plot(d_losses,label="Discriminator")
plt.legend()
plt.grid()
plt.show()

## ResNet18 Evaluation

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights=ResNet18_Weights.DEFAULT
resnet=resnet18(weights=weights).to(device)
resnet.eval()

In [ ]:
eval_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

with torch.no_grad():
    z=torch.randn(8,LATENT_DIM,1,1,device=device)
    fake=G(z).cpu()

for i in range(8):
    img=(fake[i]+1)/2
    x=eval_transform(img).unsqueeze(0).to(device)
    pred=resnet(x).softmax(dim=1)
    conf,cls=pred.max(dim=1)
    print(f'Image {i}: class={cls.item()} confidence={conf.item():.3f}')

## Student Investigation

Repeat using latent dimensions 50, 100, and 200. Compare image quality, diversity, loss curves, and ResNet18 confidence scores.